# Final Figures for the paper

In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

from src.plotting_helper.data_processing import *
from src.plotting_helper.plotting_utils import *

In [3]:
# Set project directory

PROJ_DIR = os.path.dirname(os.getcwd())
os.chdir(PROJ_DIR)
print(f"Current working directory set to {os.getcwd()}")

Current working directory set to /Users/aswathyajith/research/projects/llm_counting


In [4]:
models = ["gpt-5.2", "claude-opus-4.5", "gemini-2.5-pro"] 
DATA_SAVE_PATH_DIR = "plotting/data/how-many"
FIGS_SAVE_PATH_DIR = "plotting/figs/how-many"

## Data Processing

### Undercounting-Overcounting Asymmetry

In [40]:
from src.plotting_helper.data_processing import *

totals_by_model = pd.DataFrame()
errors_by_model = pd.DataFrame()

for model in models:
    results_path = f"results/0-shot/{model}/how_many/basic"
    save_path = f"plotting/data/how-many/{model}.csv"
    df = fetch_results(path=results_path)

    df = get_miscount_metrics(df)
    
    df["model"] = model
    cols = [
        "model",
        "input_text", 
        "target_count", 
        "model_count", 
        "output_tokens", 
        "reasoning_tokens", 
        "num_chars_label",
        "cat_label", 
        "signed_pct_err", 
        "miscount_type"
    ]
    df = df[cols].reset_index(drop=True)
    # Save to disk 
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    df.to_csv(save_path, index=False)

    totals = get_totals(df)
    errors = df[
        [
            "model", 
            "cat_label",
            "input_text",
            "miscount_type", 
            "signed_pct_err"
        ]
    ]
    totals_by_model = pd.concat([totals_by_model, totals], ignore_index=True)
    errors_by_model = pd.concat([errors_by_model, errors], ignore_index=True)

# Save totals and errors to disk


totals_path = os.path.join(DATA_SAVE_PATH_DIR, "totals.csv")
errors_path = os.path.join(DATA_SAVE_PATH_DIR, "errors.csv")
totals_by_model.to_csv(totals_path, index=False)
errors_by_model.to_csv(errors_path, index=False)


### How-Many vs List Counting

In [ ]:

model = "gemma-4-31b-it"
# model = "gpt-oss-20b:free"
plt_df = create_merged_df(model)
plt_df["signed_pct_err"] = 100 * (
    (plt_df["model_count"] - plt_df["target_count"]) / plt_df["target_count"]
)

plt_df["cat_label"] = plt_df["category"].str.upper()
save_path = f"plotting/data/how-many_vs_list/{model}.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
plt_df.to_csv(save_path, index=False)

## Plots

### Fig 2: Miscount Freq & Error by Model

In [56]:
from src.plotting_helper.plotting_utils import *

totals_by_model = pd.read_csv("plotting/data/how-many/totals.csv")
errors_by_model = pd.read_csv("plotting/data/how-many/errors.csv")
TWO_PANEL_SIZE = (3.5, 1.5)
handles, labels = [], []
metric_plt_func = {
    plot_counts_by_model: {
        "df": totals_by_model,
        "save_path": "plotting/figs/how-many/miscount_freq_by_model.pdf",
    }, 

    plot_error_magnitude_by_model: {
        "df": errors_by_model,
        "save_path": "plotting/figs/how-many/miscount_magnitude_by_model.pdf",
    }
}

for plt_func, plt_vars in metric_plt_func.items():

    with plt.rc_context(PLOT_STYLE):
        df = plt_vars["df"]
        save_path = plt_vars["save_path"]

        fig, ax, (h, l) = plt_func(
            df,
            figsize=TWO_PANEL_SIZE,
            model_label_map = MODEL_LABEL_MAP,
            model_order=MODEL_ORDER,
            hue_order=["under", "over"],
            palette=PALETTE,
            save_path=save_path
        )

        handles.extend(h)
        labels.extend(l)

priority_order = ['Overcount', 'Overcount', 'Undercount', 'Undercount', 'Total Miscounts']
handles_labels_zipped = zip(handles, labels)
handles_labels_zipped_sorted = sorted(        handles_labels_zipped, key=lambda x: priority_order.index(x[1]))
handles, labels = zip(*        handles_labels_zipped_sorted)

l = plot_legends(
    handles, 
    labels, 
    figsize=TWO_PANEL_SIZE,
    legend_save_path=f"{FIGS_SAVE_PATH_DIR}/legend_merged.pdf")

### Undercounting-Overcounting Asymmetry

In [55]:
for model in models:
    data_path = f"plotting/data/how-many/{model}.csv"
    df = pd.read_csv(data_path)
    handles, labels = [], []
    with plt.rc_context(PLOT_STYLE):
        fig1, ax1, legend1 = plot_miscount_ecdf(
            df,
            save_path=f"plotting/figs/how-many/{model}/miscount_split_by_count.pdf"
        )

        fig2, ax2, legend2 = plot_sorted_signed_pct_err(
            df,
            save_path=f"{FIGS_SAVE_PATH_DIR}/{model}/err_dist.pdf",
        )

        fig3, ax3, legend3 = plot_short_long_bar(
            df, 
            save_path=f"{FIGS_SAVE_PATH_DIR}/{model}/barplot_single_multi.pdf")
        
        handles.extend(legend1[0])
        handles.extend(legend2[0])
        labels.extend(legend1[1])
        labels.extend(legend2[1])
        # 3rd figure's legend is redundant, so we skip it
        
        priority_order = ['Overcount', 'Overcount', 'Undercount', 'Undercount', 'Cumulative split', 'Sign Boundary (B)']
        handles_labels_zipped = zip(handles, labels)
        handles_labels_zipped_sorted = sorted(        handles_labels_zipped, key=lambda x: priority_order.index(x[1]))
        handles, labels = zip(*        handles_labels_zipped_sorted)
        
        legend_fig = plot_legends(
            handles, 
            labels, 
            legend_save_path=f"{FIGS_SAVE_PATH_DIR}/{model}/legend.pdf")

        # render_inline_fig(fig1)
        # render_inline_fig(fig2)
        # render_inline_fig(fig3)
        # render_inline_fig(legend_fig)

('Overcount', 'Overcount', 'Undercount', 'Undercount', 'Cumulative split', 'Sign Boundary (B)')
('Overcount', 'Overcount', 'Undercount', 'Undercount', 'Cumulative split', 'Sign Boundary (B)')
('Overcount', 'Overcount', 'Undercount', 'Undercount', 'Cumulative split', 'Sign Boundary (B)')


In [51]:
labels

['Undercount',
 'Overcount',
 'Cumulative split',
 'Undercount',
 'Overcount',
 'Sign Boundary (B)']

dd

In [ ]:
def plot_signed_error_boxplot(
    df,
    figsize=(8, 5),
    save_path=None,
    category_order=None,
):
    plot_df = df[
        df["miscount_type"].isin(["under", "over"])
    ].copy()

    plot_df["signed_pct_err"] = (
        plot_df["signed_pct_err"]
        .replace([np.inf, -np.inf], np.nan)
    )
    plot_df = plot_df.dropna(
        subset=["signed_pct_err", "num_chars_label"]
    )

    if category_order is None:
        category_order = list(
            plot_df["num_chars_label"].drop_duplicates()
        )

    colors = {
        "under": "#4C72B0",
        "over": "#DD8452",
    }

    fig, ax = plt.subplots(figsize=figsize, layout="none")

    sns.boxplot(
        data=plot_df,
        x="num_chars_label",
        y="signed_pct_err",
        hue="miscount_type",
        order=category_order,
        hue_order=["under", "over"],
        palette=colors,
        width=0.65,
        linewidth=1,
        showfliers=False,
        ax=ax,
    )

    ax.axhline(
        0,
        color="0.3",
        linewidth=0.8,
        linestyle="--",
        zorder=0,
    )

    ax.set_xlabel("Target type")
    ax.set_ylabel(r"Signed percent error (\%)")

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles,
        ["Undercount", "Overcount"],
        title=None,
        frameon=False,
        loc="best",
    )

    fig.subplots_adjust(
        left=0.16,
        right=0.96,
        bottom=0.16,
        top=0.95,
    )

    if save_path:
        save_dir = os.path.dirname(save_path)
        if save_dir:
            os.makedirs(save_dir, exist_ok=True)

        fig.savefig(
            save_path,
            format="pdf",
        )

    return fig, ax

In [ ]:
def plot_short_long_matrix(df, save_path=None):
    count_matrix = (
        df.groupby(
            ["num_chars_label", "miscount_type"],
            observed=True,
        )
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["under", "over"], fill_value=0)
    )

    fig, ax = plt.subplots(figsize=(5, 3.5))

    sns.heatmap(
        count_matrix,
        annot=True,
        fmt=",d",
        cmap="viridis_r",
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": "Number of errors"},
        ax=ax,
    )

    ax.set_yticklabels(
        ["SINGLE\nCHARACTER\nTARGET", "MULTI\nCHARACTER\nTARGET"],
        rotation=0,
    )
    ax.set_xlabel("Miscount type")
    ax.set_ylabel("Target type")
    ax.set_xticklabels(["Undercount", "Overcount"])
    ax.tick_params(axis="y", rotation=0)

    
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path)

    plt.close()
    return fig

for model in models:
    results_path = "plotting/data/how-many/claude-opus-4.5.csv"
    df = pd.read_csv(results_path)
    save_path = f"plotting/figs/how-many/{model}/matrix_single_multi.pdf"
    fig = plot_short_long_matrix(df, save_path)
    render_inline_fig(fig)

In [133]:
for model in models:
    results_path = "plotting/data/how-many/claude-opus-4.5.csv"
    df = pd.read_csv(results_path)
    

### How-Many vs List Counting

In [ ]:
model = "gemma-4-31b-it"
data_path = f"plotting/data/how-many_vs_list/{model}.csv"
plt_df = pd.read_csv(data_path)
save_path = data_path.replace(".csv", ".pdf").replace("data", "figures")
plot_list_vs_howmany(plt_df, save_path)